# What is RoPE (Rotary Position Embedding)?

RoPE is a **positional encoding technique** used in Transformer models to inject information about the position of tokens in a sequence, but it does so differently than classic absolute or learned positional embeddings.

**Instead of adding position embeddings to token embeddings, RoPE applies a position-dependent rotation to the token embeddings in their vector space.**

This rotation encodes the relative and absolute position information directly into the query and key vectors used in the attention mechanism.

---

# Why RoPE? (The Motivation)

1. **Classic positional encoding problems:**

   - Absolute positional embeddings (like sinusoids or learned vectors) are **added** to the input token embeddings.
   - This can limit the model’s ability to generalize beyond the max sequence length seen during training.
   - It sometimes lacks an explicit relative positional encoding that helps capture how tokens relate at different distances.

2. **RoPE advantages:**

   - **Injects relative position information naturally:** Because rotations encode angles, the dot product between rotated vectors reflects relative positions.
   - **Preserves vector norms:** Rotations don’t change vector lengths, so attention scores remain stable.
   - **Generalizes better:** RoPE supports extrapolation to longer sequences than trained for.
   - **Works well with multi-head attention:** Each head’s embeddings are rotated consistently, preserving attention behavior.

---

# Mathematics of RoPE — The Core Idea

Consider a token’s vector embedding as a set of paired dimensions, e.g., dimensions $(2i, 2i+1)$ grouped together. RoPE applies a **2D rotation** in each of these paired dimensions based on the token’s position.

---

## Formula for rotation angle:

$$
\theta_{p,i} = \frac{p}{10000^{\frac{2i}{d}}}
$$

- $p$ = token position index (starting at 0)
- $i$ = pair index (which pair of dimensions)
- $d$ = total embedding dimension (usually head dimension in attention)

---

## Rotation matrix applied to each pair:

$$
\begin{bmatrix}
x'_{2i} \\
x'_{2i+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos(\theta_{p,i}) & -\sin(\theta_{p,i}) \\
\sin(\theta_{p,i}) & \cos(\theta_{p,i})
\end{bmatrix}
\begin{bmatrix}
x_{2i} \\
x_{2i+1}
\end{bmatrix}
$$

This means each pair of values in the embedding is rotated by an angle dependent on the token position and pair index.

---


# RoPE Computation Example with a Sample Word Vector

---

## Setup

- Assume a **head embedding dimension** $d = 4$ (so 2 pairs of dimensions)  
- Position $p = 2$ (the token is at index 2 in the sequence, 0-based)  
- Sample word vector (query vector for example):  
  $$
  \mathbf{x} = [1.0, 2.0, 3.0, 4.0]
  $$

---

## Step 1: Compute inverse frequencies for each pair index $i$

Recall:

$$
\text{inv\_freq}_i = \frac{1}{10000^{\frac{2i}{d}}}
$$

Calculate for $i = 0, 1$:

- $i=0$:
$$
\text{inv\_freq}_0 = \frac{1}{10000^{0}} = 1
$$

- $i=1$:
$$
\text{inv\_freq}_1 = \frac{1}{10000^{\frac{2}{4}}} = \frac{1}{10000^{0.5}} = \frac{1}{100} = 0.01
$$

---

## Step 2: Calculate rotation angles $\theta_{p,i}$

$$
\theta_{p,i} = p \times \text{inv\_freq}_i
$$

With $p = 2$:

- $i=0$:
$$
\theta_{2,0} = 2 \times 1 = 2
$$

- $i=1$:
$$
\theta_{2,1} = 2 \times 0.01 = 0.02
$$

---

## Step 3: Apply rotations to each dimension pair

Split $\mathbf{x}$ into pairs:

$$
\mathbf{x} = [x_0, x_1, x_2, x_3] = [1.0, 2.0, 3.0, 4.0]
$$

Pair 1: $[x_0, x_1] = [1.0, 2.0]$  
Pair 2: $[x_2, x_3] = [3.0, 4.0]$

---

### Pair 1 rotation with angle $\theta = 2$ radians:

$\cos(2) \approx -0.4161$, $\sin(2) \approx 0.9093$

Apply rotation matrix:

$$
\begin{bmatrix}
x'_0 \\
x'_1
\end{bmatrix}
=
\begin{bmatrix}
\cos(2) & -\sin(2) \\
\sin(2) & \cos(2)
\end{bmatrix}
\begin{bmatrix}
1.0 \\
2.0
\end{bmatrix}
$$

Calculate:

$$
x'_0 = 1.0 \times (-0.4161) - 2.0 \times 0.9093 = -0.4161 - 1.8186 = -2.2347
$$

$$
x'_1 = 1.0 \times 0.9093 + 2.0 \times (-0.4161) = 0.9093 - 0.8322 = 0.0771
$$

---

### Pair 2 rotation with angle $\theta = 0.02$ radians:

$\cos(0.02) \approx 0.9998$, $\sin(0.02) \approx 0.0200$

Apply rotation:

$$
\begin{bmatrix}
x'_2 \\
x'_3
\end{bmatrix}
=
\begin{bmatrix}
0.9998 & -0.0200 \\
0.0200 & 0.9998
\end{bmatrix}
\begin{bmatrix}
3.0 \\
4.0
\end{bmatrix}
$$

Calculate:

$$
x'_2 = 3.0 \times 0.9998 - 4.0 \times 0.0200 = 2.9994 - 0.0800 = 2.9194
$$

$$
x'_3 = 3.0 \times 0.0200 + 4.0 \times 0.9998 = 0.0600 + 3.9992 = 4.0592
$$

---

## Final rotated vector after RoPE:

$$
\mathbf{x'} = [-2.2347, 0.0771, 2.9194, 4.0592]
$$

---

# Interpretation

- Each pair in the original vector has been rotated by a different angle depending on the position and dimension.
- These rotations encode positional information directly into the vector components.
- When such rotated vectors are used in attention dot products, the positional relationships between tokens become embedded naturally.

---
